# W16-D7 实验 · Virtual CTO Review 执行版：三门体检 × 477→499 活事件复算 × docs 主题分类 × 五维双轨

与 md 的分工：md 是评审记录与裁决（阅读材料）；本 ipynb 是**今晨全部证据的可执行复算**——

1. §1 三门体检：subprocess 实跑 G-05 / Identity / canonical 三门，断言汇总（含 canonical 门 RED=执勤中）
2. §2 活事件：`git show origin/main` 集合 diff 复算 477→499，五族构成断言 + 扩案时间线图
3. §3 docs 增量主题分类（备份线占比）+ 探针错仓实证（真值 24 vs 假值 87）
4. §4 五维双轨趋势（均值/木桶两口径）+ 预填 7.0 vs 复算 6.9 校准断言

断言即证据：任何一条 FAIL = 本 Review 结论失效。

In [ ]:
# ---- 环境：中文字体（TOOLS.md 标准方式）+ 路径 ----
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

import subprocess, sys, json
from pathlib import Path
from collections import Counter

BASE = Path("/root/learning-notebooks")
GOV = BASE / "semantic-model" / "governance"
W16D = BASE / "第16周"

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr

## §1 三门体检：W16 治理地基的今晨实跑

三门同哲学（fail-closed / 三级判决 / --change 承认 / --json 机读）。期望：
G-05 anchors GREEN 15/15、Identity verify GREEN 26/26 + selftest GREEN、
canonical verify **RED 44 BROKEN（执勤中，非失败——它在咬真骨头）** + selftest GREEN。

In [ ]:
# ---- §1 三门体检（真实验证，非转述）----
G05 = GOV / "ci" / "frozen_effect_ci.py"
IDG = GOV / "ci" / "identity_anchor_ci.py"
CDG = GOV / "ci" / "canonical_drift_ci.py"

rc_g05, out_g05 = run([sys.executable, str(G05), "anchors",
                       "--anchors", str(GOV / "g05-effect-anchors.yaml"),
                       "--repo", "/root/lnkcre"])
line_g05 = out_g05.strip().splitlines()[-1]
assert rc_g05 == 0 and "GREEN" in line_g05 and "OK 15" in line_g05, line_g05

rc_idv, out_idv = run([sys.executable, str(IDG), "verify",
                       "--anchors", str(GOV / "identity-analysis-anchors.yaml")])
line_idv = out_idv.strip().splitlines()[-1]
rc_ids, out_ids = run([sys.executable, str(IDG), "selftest",
                       "--anchors", str(GOV / "identity-analysis-anchors.yaml")])
assert rc_idv == 0 and "OK 26" in line_idv, line_idv
assert rc_ids == 0 and "GREEN" in out_ids, out_ids[-200:]

rc_cd, out_cd = run([sys.executable, str(CDG), "verify",
                     "--baseline", str(GOV / "canonical-baseline-w39.txt"), "--json"])
GATE_SUMMARY = json.loads(out_cd)["summary"]
rc_cds, out_cds = run([sys.executable, str(CDG), "selftest"])
assert rc_cds == 0 and "GREEN" in out_cds, out_cds[-200:]
assert GATE_SUMMARY["verdict"] == "RED"
assert GATE_SUMMARY["added"] == 22 and GATE_SUMMARY["removed"] == 0
assert GATE_SUMMARY["broken"] == 44 and GATE_SUMMARY["cited"] == 0

rows = [
    ("G-05 frozen (anchors)",   line_g05.split("→")[-1].strip(), "15 锚零漂移"),
    ("Identity (verify)",       line_idv.split("→")[-1].strip(), "20 正 + 6 负锚"),
    ("Identity selftest",       "GREEN", "碰撞哨兵 PASS（突变留痕）"),
    ("canonical drift (verify)","RED（执勤中）44 BROKEN", "22 孤儿 + 22 泄漏"),
    ("canonical selftest",      "GREEN", "6/6 突变全过（含等量换血）"),
]
print("=== W16-D7 三门体检（实跑复算）===")
for name, verdict, note in rows:
    print("  {:26s} {:26s} {}".format(name, verdict, note))
print("canonical 现实源:", GATE_SUMMARY["reality"]["source"],
      "| 表数:", GATE_SUMMARY["reality"]["tables"],
      "| 集合指纹:", GATE_SUMMARY["reality"]["set_sha256_16"])

## §2 活事件复算：477 → 484 → 499（五族 22 表）

- 集合 diff 用 `git show origin/main:…testdata/canonical_tables.txt` 直读现实源（499）
- 基线 = `canonical-baseline-w39.txt`（477，D6 冻结，source_rev 0392e107）
- 家族按名前缀分组（五族恰好全覆盖 22 表，断言无漏）；到达波次归因以门输出 file:line 为准
- 注意同名双文件陷阱：`migrations/canonical_tables.txt` 只有 3 表（con-005 局部集），全量表在 `testdata/`

In [ ]:
# ---- §2 集合 diff 复算 + 五族构成 + 扩案时间线 ----
CT_PATH = "backend/internal/platform/database/testdata/canonical_tables.txt"
origin_raw = subprocess.run(["git", "-C", "/root/lnkcre", "show", "origin/main:" + CT_PATH],
                            capture_output=True, text=True).stdout
origin_set = set(l.strip() for l in origin_raw.splitlines() if l.strip() and not l.startswith("#"))
base_set = set(l.strip() for l in open(GOV / "canonical-baseline-w39.txt")
               if l.strip() and not l.startswith("#"))
added = sorted(origin_set - base_set)
removed = sorted(base_set - origin_set)
assert len(base_set) == 477 and len(origin_set) == 499
assert len(added) == 22 and removed == []

FAMILIES = {
    "indicator-target": {"indicator_monthly_target_history", "indicator_monthly_targets", "target_indicators"},
    "leasing-progress": {"leasing_progress_tasks", "leasing_progress_settings",
                         "leasing_progress_setting_audits", "leasing_progress_task_adjustments",
                         "leasing_progress_event_consumptions", "leasing_stage_templates",
                         "leasing_stage_template_items", "leasing_stage_completion_facts",
                         "leasing_stage_plan_overrides", "leasing_plan_recalc_batches"},
    "unit-leasing": {"unit_leasing_plans", "unit_leasing_stages"},
    "leasing-policy": {"leasing_policies", "leasing_policy_versions",
                       "leasing_policy_lifecycle", "price_authority_constraints"},
    "unit-pricing": {"unit_pricing_batches", "unit_pricing_batch_lines", "unit_pricing_batch_ops"},
}
uncovered = set(added) - set().union(*FAMILIES.values())
assert not uncovered, uncovered
fam_counts = {k: len(v & set(added)) for k, v in FAMILIES.items()}
assert sum(fam_counts.values()) == 22
print("五族构成:", fam_counts)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
days = ["09-16 D3\n基线冻结", "09-18 D5\n活事件首报", "09-19 D6", "09-20 D7\n今日"]
totals = [477, 484, 484, 499]
bars = ax1.bar(days, totals, color=["#9e9e9e", "#ff9800", "#ff9800", "#e53935"])
for b, t, d in zip(bars, totals, [0, 7, 0, 15]):
    ax1.annotate(str(t), (b.get_x() + b.get_width() / 2, t), ha="center", va="bottom", fontsize=10)
    if d:
        ax1.annotate("+" + str(d), (b.get_x() + b.get_width() / 2, t - 40),
                     ha="center", va="top", fontsize=9, color="#b71c1c", fontweight="bold")
ax1.set_title("canonical 表宇宙扩案时间线（集合门实测）")
ax1.set_ylabel("表数")
ax1.set_ylim(460, 510)

names = list(fam_counts.keys())
vals = [fam_counts[n] for n in names]
ax2.barh(names, vals, color="#1565c0")
for i, v in enumerate(vals):
    ax2.annotate(str(v), (v, i), va="center", ha="left", fontsize=10)
ax2.set_title("D7 新增 22 表的五族构成（名前缀分组）")
ax2.set_xlabel("表数")
ax2.set_xlim(0, 12)
plt.tight_layout()
plt.savefig(W16D / "w16d7_canonical_499.png", dpi=130)
plt.close()
print("图已保存: w16d7_canonical_499.png")

## §3 docs 增量主题分类 + 探针错仓实证

- docs 增量 24 commits 主题 = 备份线 15 / mi-cre 收口 4 / lnkchat 受理 4 / lnkreport 1
- **断言零 ontology/governance 主题** → G-01 第 3 周未受理的证据（受理机器活着：同窗 accept 了 E1/F1/DV-LC-004）
- 错仓实证：`/root/langchat-docs` 同样含 lanlnk/ 目录树（business-ontology.yaml 同指纹副本），
  人肉 cd 探针命中它得假值 87；真仓 `/root/docs` = 24 → S2 脚本化的直接理由

In [ ]:
# ---- §3 docs 主题分类（真实 git log 解析）+ 错仓对比 ----
log = subprocess.run(["git", "-C", "/root/docs", "log", "--oneline", "HEAD..origin/main"],
                     capture_output=True, text=True).stdout
lines = [l for l in log.splitlines() if l.strip()]

def theme(msg):
    if msg.startswith("docs: 备份") or "chore(backup)" in msg or "chore(docs)" in msg:
        return "备份线"
    if "(mi-cre)" in msg or "(lnkcre)" in msg:
        return "mi-cre 收口"
    if "(lnkchat)" in msg:
        return "lnkchat 受理"
    if "(lnkreport)" in msg:
        return "lnkreport"
    return "其他"

cnt = Counter(theme(l.split(" ", 1)[1]) for l in lines)
ontology_hits = [l for l in lines if "ontology" in l.lower() or "governance-header" in l]
assert len(lines) == 24, len(lines)
assert not ontology_hits, ontology_hits
assert cnt["备份线"] == 15 and cnt["mi-cre 收口"] == 4 and cnt["lnkchat 受理"] == 4 and cnt["lnkreport"] == 1

behind_true = subprocess.run(["git", "-C", "/root/docs", "rev-list", "--count", "HEAD..origin/main"],
                             capture_output=True, text=True).stdout.strip()
behind_fake = subprocess.run(["git", "-C", "/root/langchat-docs", "rev-list", "--count", "HEAD..origin/main"],
                             capture_output=True, text=True).stdout.strip()
assert behind_true == "24"

fig, ax = plt.subplots(figsize=(7.2, 4))
labels = list(cnt.keys())
sizes = [cnt[k] for k in labels]
colors = ["#9e9e9e", "#2e7d32", "#1565c0", "#6a1b9a", "#ef6c00"][: len(labels)]
wedges, txts, autotxts = ax.pie(sizes, labels=labels, colors=colors, autopct="%1.0f%%",
                                startangle=90, counterclock=False)
ax.set_title("docs 增量 24 commits 主题分类（备份线 15 = 62.5%，零 ontology 主题）")
plt.tight_layout()
plt.savefig(W16D / "w16d7_docs_themes.png", dpi=130)
plt.close()
print("主题分布:", dict(cnt))
print("真仓 /root/docs behind =", behind_true, "| 错仓 /root/langchat-docs behind =", behind_fake,
      "（同含 lanlnk 目录树 → 人肉探针陷阱）")
print("图已保存: w16d7_docs_themes.png")

## §4 五维双轨趋势 + 预填校准（评分纪律 v1.1：发布必带口径）

- 均值口径：(7.5+7.0+6.5+7.5+6.0)/5 = **6.9**（W15 6.9 持平）
- 木桶口径：**6.0**（W15 6.5 回摆 0.5——DX 单维下调 0.5：探针人肉踩坑实锤 + 消费面零新动作）
- W14 用 W15-D7 复算值（发布 6.4 无口径不可复算，已废）
- 预填 7.0（D5）vs 复算 6.9：|Δ|=0.1 ≤ 0.3 → 校准履历样本 3 **命中**

In [ ]:
# ---- §4 五维趋势复算与断言 ----
weeks = ["W14", "W15", "W16"]
mean_track = [6.8, 6.9, 6.9]     # 均值口径（W14 为 W15-D7 复算值）
bucket_track = [5.5, 6.5, 6.0]   # 木桶口径
dims16 = {"Architecture\nQuality": 7.5, "Code\nHealth": 7.0, "ADR\nConsistency": 6.5,
          "Technical\nDebt": 7.5, "Developer\nExperience": 6.0}
assert abs(sum(dims16.values()) / 5 - 6.9) < 1e-9
assert min(dims16.values()) == 6.0
prefill, final = 7.0, 6.9
assert abs(prefill - final) <= 0.3

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
ax1.plot(weeks, mean_track, "o-", color="#1565c0", label="均值口径")
ax1.plot(weeks, bucket_track, "s--", color="#e53935", label="木桶口径")
for x, y in zip(weeks, mean_track):
    ax1.annotate(str(y), (x, y), textcoords="offset points", xytext=(0, 8), ha="center", color="#1565c0")
for x, y in zip(weeks, bucket_track):
    ax1.annotate(str(y), (x, y), textcoords="offset points", xytext=(0, -14), ha="center", color="#e53935")
ax1.scatter(["W16"], [prefill], marker="*", s=200, color="#ff9800", zorder=5,
            label="W16 预填 7.0（D5）")
ax1.annotate("DX 6.5→6.0\n探针坑实锤", ("W16", 6.0), textcoords="offset points",
             xytext=(-6, -30), ha="right", fontsize=9, color="#b71c1c")
ax1.set_ylim(4.8, 7.8)
ax1.set_title("五维评分双轨趋势（发布口径：均值 6.9 / 木桶 6.0）")
ax1.set_ylabel("分")
ax1.legend(loc="lower right", fontsize=9)
ax1.grid(alpha=0.3)

names = list(dims16.keys())
vals = [dims16[n] for n in names]
barcolors = ["#2e7d32" if v >= 7.5 else "#1565c0" if v >= 6.5 else "#e53935" for v in vals]
ax2.bar(names, vals, color=barcolors)
for i, v in enumerate(vals):
    ax2.annotate(str(v), (i, v), ha="center", va="bottom", fontsize=10)
ax2.axhline(6.9, color="#1565c0", ls=":", lw=1)
ax2.annotate("均值 6.9", (4.4, 6.95), fontsize=9, color="#1565c0", ha="right")
ax2.set_ylim(0, 8.6)
ax2.set_title("W16 五维终值（木桶 = DX 6.0）")
ax2.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(W16D / "w16d7_score_trend.png", dpi=130)
plt.close()
print("W16 终值: 均值 6.9（持平 W15）/ 木桶 6.0（回摆 0.5）")
print("预填 7.0 vs 复算 6.9 → |Δ|=0.1 ≤ 0.3，校准命中（履历样本 3：命中上沿 / MISS 偏高 / 命中）")
print("图已保存: w16d7_score_trend.png")

## 结论（与 ipynb 断言一一对应）

| 断言 | 复算结果 |
|---|---|
| 三门体检 | G-05 15/15 GREEN · Identity 26/26 GREEN + selftest · canonical selftest 6/6 |
| canonical 门执勤 | RED：added 22 / removed 0 / leaked 22 / BROKEN 44 / cited 0（现实源 lnkcre@origin/main 499 表） |
| 五族全覆盖 | indicator-target 3 + leasing-progress 10 + unit-leasing 2 + leasing-policy 4 + unit-pricing 3 = 22 |
| docs 增量 | 24 commits：备份 15 / mi-cre 4 / lnkchat 4 / lnkreport 1，**零 ontology 主题**（G-01 第 3 周未受理） |
| 探针错仓 | 真仓 24 vs 错仓 87（langchat-docs 同含 lanlnk 目录树）→ S2 脚本化 W17-D1 |
| 五维 | 均值 6.9 / 木桶 6.0 / 预填校准命中（Δ0.1） |

全部断言通过 = 本 Review 的证据链闭合。W17 定轨见 md §7。